# 3.29 — Support Vector Machines & Margins

Support Vector Machines choose a separating hyperplane by asking for the widest safe buffer between classes, not merely for a line that classifies the training points. In this lesson we build the score, margin, hinge loss, slack variables, and support-vector idea directly in NumPy so the optimization formula $$\min_w \tfrac12\|w\|^2 + C\sum_i \xi_i$$ feels like arithmetic you can inspect rather than magic.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build support vector machines one idea at a time. Run each cell in order and read the printed intermediate values — every geometric quantity is made visible. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, dot products, norms, and small grid searches.
import matplotlib.pyplot as plt  # visualizations for hyperplanes, margins, and losses.
np.random.seed(0)  # reproducibility for any random demos.

### 1. A linear score turns points into signed decisions

An SVM begins with a linear score $$f(x)=w^\top x+b$$. The sign of this score is the predicted class: positive points should land above zero and negative points below zero. Before talking about margins, we need to see the separator as a concrete line and the scores as signed distances up to a scale.

In [ ]:
X_w = np.array([[1., 1.], [2., 1.], [1., 2.], [0., 0.], [-1., 0.], [0., -1.]])  # six separable 2-D points.
y_w = np.array([1., 1., 1., -1., -1., -1.])  # labels encoded as +1 and -1 for margin algebra.
w_w = np.array([1., 1.])  # a simple normal vector pointing toward the positive class.
b_w = -1.0  # shift the separating line w·x+b=0 away from the origin.
scores_w = X_w @ w_w + b_w  # signed raw scores for every point.

print("scores:", scores_w)  # inspect which side of the line each point occupies.
print("predictions:", np.sign(scores_w).astype(int))  # sign gives the class decision.

assert np.all(np.sign(scores_w) == y_w)  # this particular line separates the toy data.

▶ What you'll see: positive examples have positive scores and negative examples have negative scores.

In [ ]:
xx_w = np.linspace(-3, 4, 100)  # x-axis values for drawing the line.
yy_w = -(w_w[0] * xx_w + b_w) / w_w[1]  # solve w0*x + w1*y + b = 0 for y.
plt.figure(figsize=(4.6, 3.6))
plt.scatter(X_w[y_w == 1, 0], X_w[y_w == 1, 1], color="seagreen", label="+1")
plt.scatter(X_w[y_w == -1, 0], X_w[y_w == -1, 1], color="crimson", label="-1")
plt.plot(xx_w, yy_w, color="black", label="w·x+b=0")
plt.axis("equal"); plt.legend(); plt.title("1: a separating hyperplane"); plt.show()

▶ What you'll see: a straight line cleanly separates the two classes.

*Why it's done this way:* encoding labels as ±1 makes correctness a single product: $$y_i(w^\top x_i+b)>0$$. The vector $w$ is perpendicular to the line, so moving in the direction of $w$ increases the score; the bias $b$ shifts the line without changing its orientation.

### 2. Functional margin is confidence on the correct side

The raw score alone is not enough because a negative example with score `-4` is confidently correct, not badly wrong. Multiplying by the label gives the **functional margin** $$m_i=y_i(w^\top x_i+b)$$. Correct and confident points have large positive margins; wrong points have negative margins.

In [ ]:
functional_w = y_w * scores_w  # signed correctness margin for each training example.

print("functional margins:", functional_w)  # all positive means every point is correctly classified.
print("smallest functional margin:", functional_w.min())  # the weakest point controls the margin story.

assert functional_w.min() == 1.0  # the closest points sit exactly on the canonical margin.

▶ What you'll see: all margins are positive, but the smallest margin is only 1.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(np.arange(len(functional_w)), functional_w, color=np.where(y_w > 0, "seagreen", "crimson"))
plt.axhline(1, color="black", linestyle="--", label="canonical target m=1")
plt.xlabel("training point"); plt.ylabel("y(w·x+b)"); plt.title("2: functional margins"); plt.legend(); plt.show()

▶ What you'll see: the bars at height 1 are the tight examples; they are the first candidates to become support vectors.

*Why it's done this way:* multiplying by $y_i$ folds both classes onto the same scale. The SVM constraint $$y_i(w^\top x_i+b)\ge 1$$ says every point must be on the correct side with at least one unit of functional margin, so the weakest point defines whether the separator is acceptable.

### 3. Geometric margin divides out arbitrary scaling

If we multiply $w$ and $b$ by 10, every functional margin also multiplies by 10, but the line itself does not move. The true Euclidean distance to the decision boundary is the **geometric margin** $$\gamma_i=\frac{y_i(w^\top x_i+b)}{\|w\|}$$, which removes that arbitrary scaling.

In [ ]:
norm_w = np.linalg.norm(w_w)  # length of the normal vector.
geometric_w = functional_w / norm_w  # signed distances to the boundary for this scaled separator.

print("||w||:", round(norm_w, 3))
print("geometric margins:", np.round(geometric_w, 3))
print("minimum geometric margin:", round(float(geometric_w.min()), 3))

assert round(float(geometric_w.min()), 3) == 0.707

▶ What you'll see: the closest points are 0.707 units from the separating line.

In [ ]:
scale_w = 10.0  # multiply the entire score by a large constant.
functional_scaled_w = y_w * (X_w @ (scale_w * w_w) + scale_w * b_w)  # functional margins inflate.
geometric_scaled_w = functional_scaled_w / np.linalg.norm(scale_w * w_w)  # geometric margins do not.

print("functional before/after scaling:", functional_w.min(), functional_scaled_w.min())
print("geometric before/after scaling:", round(float(geometric_w.min()), 3), round(float(geometric_scaled_w.min()), 3))

assert np.allclose(geometric_w, geometric_scaled_w)

▶ What you'll see: functional margins change from 1 to 10, while geometric margins stay the same.

In [ ]:
yy_scaled_w = -((scale_w * w_w)[0] * xx_w + scale_w * b_w) / (scale_w * w_w)[1]  # scaled separator, same geometry.
upper_geom_w = (1 - b_w - w_w[0] * xx_w) / w_w[1]  # original +1 canonical margin.
lower_geom_w = (-1 - b_w - w_w[0] * xx_w) / w_w[1]  # original -1 canonical margin.
closest_geom_w = geometric_w <= geometric_w.min() + 1e-12  # points at the minimum geometric margin.
plt.figure(figsize=(4.8, 3.8))
plt.scatter(X_w[y_w == 1, 0], X_w[y_w == 1, 1], color="seagreen", label="+1")
plt.scatter(X_w[y_w == -1, 0], X_w[y_w == -1, 1], color="crimson", label="-1")
plt.scatter(X_w[closest_geom_w, 0], X_w[closest_geom_w, 1], s=180, facecolors="none", edgecolors="black", linewidths=2, label="min distance")
plt.plot(xx_w, yy_w, color="black", label="original f=0")
plt.plot(xx_w, yy_scaled_w, ":", color="purple", linewidth=3, label="scaled f=0")
plt.plot(xx_w, upper_geom_w, "--", color="gray", label="canonical margins")
plt.plot(xx_w, lower_geom_w, "--", color="gray")
plt.axis("equal"); plt.legend(); plt.title("3: geometric margin ignores score scaling"); plt.show()

▶ What you'll see: the scaled boundary overlays the original line, while the circled closest points keep the same geometric distance to the margin street.

*Why it's done this way:* without dividing by $\|w\|$, a model could look more confident by multiplying all parameters by a constant. Geometric margin is the real distance, so maximizing it is a meaningful geometric goal rather than a scaling trick.

### 4. Maximizing margin becomes minimizing ||w||

SVMs choose the separator with the largest minimum distance to the training points. In the canonical scaling where the tightest functional margin is 1, the margin width on each side is $$1/\|w\|$$ and the full street between classes is $$2/\|w\|$$. Therefore maximizing the margin is equivalent to minimizing $$\tfrac12\|w\|^2$$ subject to the margin constraints.

In [ ]:
candidates_w = [(np.array([1., 1.]), -1.0), (np.array([2., 1.]), -1.0), (np.array([1., 2.]), -1.0), (np.array([0.8, 0.8]), -0.8)]  # candidate separators.
rows_w = []  # collect norm, min functional margin, and geometric margin.
for wc_w, bc_w in candidates_w:
    mf_w = np.min(y_w * (X_w @ wc_w + bc_w))  # tightest functional margin.
    mg_w = mf_w / np.linalg.norm(wc_w)  # tightest geometric margin.
    rows_w.append([np.linalg.norm(wc_w), mf_w, mg_w])

print("columns: ||w||, min functional, min geometric")
print(np.round(np.array(rows_w), 3))

assert round(rows_w[0][2], 3) == 0.707

▶ What you'll see: different separating lines can have different geometric margins even when all classify correctly.

In [ ]:
norms_w = np.array([r[0] for r in rows_w])
margins_w = np.array([r[2] for r in rows_w])
plt.figure(figsize=(4.8, 3))
plt.scatter(norms_w, margins_w, color="purple")
for j_w, (n_w, m_w) in enumerate(zip(norms_w, margins_w)):
    plt.text(n_w + 0.02, m_w, f"cand {j_w}")
plt.xlabel("||w||"); plt.ylabel("minimum geometric margin"); plt.title("4: smaller norm usually means wider margin"); plt.show()

▶ What you'll see: the separator with the smaller effective norm has the wider safety buffer.

*Why it's done this way:* the hard-margin problem fixes the closest functional margin at 1 to remove scale ambiguity. After that choice, only $\|w\|$ controls the actual distance, so minimizing $\tfrac12\|w\|^2$ is the algebraic version of making the margin street as wide as possible.

### 5. Hinge loss and slack allow violations

Real data is rarely perfectly separable. The soft-margin SVM introduces slack variables $$\xi_i=\max(0,1-y_i(w^\top x_i+b))$$, which are exactly the hinge losses. A point outside the margin has zero slack; a point inside the margin has positive slack; a misclassified point has slack greater than 1.

In [ ]:
X_soft_w = X_w.copy()
y_soft_w = y_w.copy()
X_soft_w[2] = np.array([0.2, 0.1])  # move one positive point close to the negative side.
scores_soft_w = X_soft_w @ w_w + b_w  # reuse the same separator.
margins_soft_w = y_soft_w * scores_soft_w  # functional margins with one difficult point.
slack_w = np.maximum(0, 1 - margins_soft_w)  # hinge loss equals margin violation.

print("margins:", np.round(margins_soft_w, 3))
print("slack / hinge:", np.round(slack_w, 3))

assert round(float(slack_w[2]), 3) == 1.7

▶ What you'll see: the moved positive point has large slack because it violates the desired margin.

In [ ]:
grid_m_w = np.linspace(-1.5, 3.0, 100)  # possible functional margins.
hinge_curve_w = np.maximum(0, 1 - grid_m_w)  # hinge loss curve.
plt.figure(figsize=(4.6, 3))
plt.plot(grid_m_w, hinge_curve_w, color="navy")
plt.scatter(margins_soft_w, slack_w, color="crimson", zorder=3)
plt.axvline(1, color="black", linestyle="--", label="margin target")
plt.xlabel("functional margin y(w·x+b)"); plt.ylabel("hinge loss"); plt.title("5: hinge charges only margin violations"); plt.legend(); plt.show()

▶ What you'll see: loss is zero at margin 1 or above and rises linearly below that.

*Why it's done this way:* the constraint form $$y_i(w^\top x_i+b)\ge 1-\xi_i$$ becomes a loss because the smallest feasible slack is $$\max(0,1-y_if(x_i))$$. The objective $$\tfrac12\|w\|^2+C\sum_i\xi_i$$ then balances a wide margin against the cost of violations.

### 6. The C knob trades margin width against mistakes

The constant $C$ says how expensive violations are. Small $C$ is forgiving and prefers a simple wide-margin separator; large $C$ fights hard to reduce hinge loss, even if the norm grows. We can see that tradeoff by scoring a few candidate lines with the soft-margin objective.

In [ ]:
soft_candidates_w = [(np.array([1., 1.]), -1.0), (np.array([1.8, 1.8]), -1.0), (np.array([0.6, 0.6]), -0.6)]  # candidate soft-margin separators.
Cs_w = np.array([0.1, 1.0, 10.0])  # forgiving to strict violation penalties.
objective_w = np.zeros((len(Cs_w), len(soft_candidates_w)))  # rows C, columns candidate.
for c_idx_w, C_w in enumerate(Cs_w):
    for j_w, (wc_w, bc_w) in enumerate(soft_candidates_w):
        margins_c_w = y_soft_w * (X_soft_w @ wc_w + bc_w)
        hinge_c_w = np.maximum(0, 1 - margins_c_w).sum()
        objective_w[c_idx_w, j_w] = 0.5 * np.dot(wc_w, wc_w) + C_w * hinge_c_w

print("objectives rows C=0.1,1,10:")
print(np.round(objective_w, 3))

▶ What you'll see: as C changes, the best objective can move to a different separator.

In [ ]:
best_idx_w = np.argmin(objective_w, axis=1)  # best candidate for each C.
plt.figure(figsize=(5, 3))
for j_w in range(objective_w.shape[1]):
    plt.plot(Cs_w, objective_w[:, j_w], marker="o", label=f"candidate {j_w}")
plt.xscale("log"); plt.xlabel("C"); plt.ylabel("0.5||w||² + C∑hinge"); plt.title("6: C changes what the objective values"); plt.legend(); plt.show()

print("best candidate by C:", best_idx_w)

▶ What you'll see: strict C makes hinge violations dominate, while small C makes norm control dominate.

*Why it's done this way:* $C$ puts margin violations and model complexity on the same accounting sheet. If violations are cheap, the optimizer accepts some slack to keep $\|w\|$ small; if violations are expensive, it bends toward fitting difficult points.

### 7. Support vectors are the points that touch or violate the margin

After fitting, most points do not affect the chosen boundary because they are safely beyond the margin. The important points are those with $$y_i(w^\top x_i+b)\le 1$$: they touch the margin or violate it. These are the **support vectors**.

In [ ]:
support_mask_w = functional_w <= 1 + 1e-12  # points on the canonical margin for the hard-margin toy separator.
support_idx_w = np.where(support_mask_w)[0]  # indices of support vectors.

print("support vector indices:", support_idx_w)
print("support vector coordinates:\n", X_w[support_idx_w])

assert support_idx_w.tolist() == [0, 3]

▶ What you'll see: only two points exactly touch the margin in this tiny hard-margin example.

In [ ]:
upper_w = (1 - b_w - w_w[0] * xx_w) / w_w[1]  # w·x+b=+1 margin line.
lower_w = (-1 - b_w - w_w[0] * xx_w) / w_w[1]  # w·x+b=-1 margin line.
plt.figure(figsize=(4.8, 3.8))
plt.scatter(X_w[y_w == 1, 0], X_w[y_w == 1, 1], color="seagreen", label="+1")
plt.scatter(X_w[y_w == -1, 0], X_w[y_w == -1, 1], color="crimson", label="-1")
plt.scatter(X_w[support_mask_w, 0], X_w[support_mask_w, 1], s=180, facecolors="none", edgecolors="black", linewidths=2, label="support")
plt.plot(xx_w, yy_w, color="black"); plt.plot(xx_w, upper_w, "--", color="gray"); plt.plot(xx_w, lower_w, "--", color="gray")
plt.axis("equal"); plt.legend(); plt.title("7: support vectors define the margin"); plt.show()

▶ What you'll see: the circled points lie on the dashed margin lines, so moving them would move the boundary.

*Why it's done this way:* the max-margin solution is controlled by the active constraints. Points with margin greater than 1 have slack in the inequalities, so small changes to them do not change the optimal separator; support vectors are the constraints that are tight.

### 8. A tiny grid search makes the objective tangible

Full SVM solvers use convex optimization, but a small grid search is enough to see the objective choose a line. We search over a few slopes and biases, compute hinge loss plus the norm penalty, and keep the lowest-scoring separator.

In [ ]:
w0_grid_w = np.linspace(0.4, 1.6, 13)  # candidate first weight.
w1_grid_w = np.linspace(0.4, 1.6, 13)  # candidate second weight.
b_grid_w = np.linspace(-2.0, 0.0, 17)  # candidate bias values.
C_grid_w = 1.0  # moderate soft-margin penalty.
best_obj_w = np.inf
best_tuple_w = None
for w0_w in w0_grid_w:
    for w1_w in w1_grid_w:
        for bg_w in b_grid_w:
            wg_w = np.array([w0_w, w1_w])
            m_w = y_w * (X_w @ wg_w + bg_w)
            obj_w = 0.5 * np.dot(wg_w, wg_w) + C_grid_w * np.maximum(0, 1 - m_w).sum()
            if obj_w < best_obj_w:
                best_obj_w = obj_w
                best_tuple_w = (wg_w.copy(), bg_w, m_w.copy())

print("best w, b, objective:", np.round(best_tuple_w[0], 3), round(best_tuple_w[1], 3), round(best_obj_w, 3))

assert best_obj_w < 1.2

▶ What you'll see: the grid chooses a separator close to the intuitive diagonal boundary.

In [ ]:
best_w_w, best_b_w, best_margins_w = best_tuple_w
best_support_w = best_margins_w <= 1 + 1e-9

print("best margins:", np.round(best_margins_w, 3))
print("support count:", int(best_support_w.sum()))

plt.figure(figsize=(4.8, 3.8))
plt.scatter(X_w[y_w == 1, 0], X_w[y_w == 1, 1], color="seagreen")
plt.scatter(X_w[y_w == -1, 0], X_w[y_w == -1, 1], color="crimson")
yline_w = -(best_w_w[0] * xx_w + best_b_w) / best_w_w[1]
plt.plot(xx_w, yline_w, color="black", label="best grid line")
plt.scatter(X_w[best_support_w, 0], X_w[best_support_w, 1], s=180, facecolors="none", edgecolors="black")
plt.axis("equal"); plt.legend(); plt.title("8: grid-search SVM objective"); plt.show()

▶ What you'll see: the selected separator is governed by a small set of margin-touching points.

*Why it's done this way:* the grid search is intentionally crude, but the scoring rule is the real SVM objective. Every candidate pays for both complexity and margin violations, so the selected line is the one with the best full decision score, not merely the prettiest training split.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per mechanic in this lesson. Each uses a handful of small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Scores become signs

A linear SVM score is just `w·x + b`. The sign of that score gives the class decision.

In [ ]:
import numpy as np                              # arrays, dot products, and checks.
import matplotlib.pyplot as plt                 # one picture per toy.

t1_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t1_X = np.array([[2.0, 1.0], [1.0, 2.0], [2.0, 2.0], [-1.0, -1.0], [-2.0, -1.0], [-1.0, -2.0]])
t1_y = np.array([1.0, 1.0, 1.0, -1.0, -1.0, -1.0])
t1_w = np.array([1.0, 1.0])
t1_b = -0.5
t1_scores = t1_X @ t1_w                         # -> [3.0, 3.0, 4.0, -2.0, -3.0, -3.0]
t1_scores = t1_scores + t1_b                    # -> [2.5, 2.5, 3.5, -2.5, -3.5, -3.5]
t1_pred = np.sign(t1_scores).astype(int)        # -> [1, 1, 1, -1, -1, -1]

print("X:", t1_X.tolist())                     # -> [[2.0, 1.0], [1.0, 2.0], [2.0, 2.0], [-1.0, -1.0], [-2.0, -1.0], [-1.0, -2.0]]
print("y:", t1_y.astype(int).tolist())          # -> [1, 1, 1, -1, -1, -1]
print("w:", t1_w.tolist())                     # -> [1.0, 1.0]
print("b:", t1_b)                              # -> -0.5
print("scores:", t1_scores.tolist())           # -> [2.5, 2.5, 3.5, -2.5, -3.5, -3.5]
print("predictions:", t1_pred.tolist())        # -> [1, 1, 1, -1, -1, -1]

assert np.array_equal(t1_pred, t1_y.astype(int))

t1_grid = np.linspace(-3.0, 3.0, 80)
t1_line = -(t1_w[0] * t1_grid + t1_b) / t1_w[1]
plt.figure(figsize=(4.6, 3.2))
plt.scatter(t1_X[t1_y == 1, 0], t1_X[t1_y == 1, 1], color="seagreen", label="+1")
plt.scatter(t1_X[t1_y == -1, 0], t1_X[t1_y == -1, 1], color="crimson", label="-1")
plt.plot(t1_grid, t1_line, color="black", label="score=0")
plt.axis("equal")
plt.legend()
plt.title("Toy 1 · sign(w·x+b) classifies")
plt.show()

▶ What you'll see: all six signs match the labels, with the black line between the two clouds.

### ✍️ Toy 2 · Functional margins fold the label into confidence

Multiplying a raw score by the label makes both classes comparable: larger positive values mean
more confident correct-side predictions.

In [ ]:
import numpy as np                              # arrays and elementwise products.

t2_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t2_y = np.array([1.0, 1.0, -1.0, -1.0, 1.0, -1.0])
t2_scores = np.array([2.0, 0.5, -1.5, -0.25, 1.2, -2.5])
t2_functional = t2_y * t2_scores                # -> [2.0, 0.5, 1.5, 0.25, 1.2, 2.5]
t2_min_margin = float(t2_functional.min())      # -> 0.25

print("labels:", t2_y.astype(int).tolist())     # -> [1, 1, -1, -1, 1, -1]
print("raw scores:", t2_scores.tolist())        # -> [2.0, 0.5, -1.5, -0.25, 1.2, -2.5]
print("functional margins:", t2_functional.tolist())  # -> [2.0, 0.5, 1.5, 0.25, 1.2, 2.5]
print("smallest margin:", t2_min_margin)       # -> 0.25

assert t2_min_margin == 0.25

plt.figure(figsize=(4.6, 2.8))
plt.bar(np.arange(t2_functional.size), t2_functional, color=np.where(t2_functional >= 1.0, "teal", "orange"))
plt.axhline(1.0, color="black", linestyle="--", label="target margin 1")
plt.xlabel("example")
plt.ylabel("y·score")
plt.legend()
plt.title("Toy 2 · weak points have margin < 1")
plt.show()

▶ What you'll see: examples 1 and 3 are correct but inside the desired margin street.

### ✍️ Toy 3 · Geometric margins ignore scaling

Functional margins inflate if you multiply every score by a constant. Dividing by `||w||` makes the
actual distance unchanged.

In [ ]:
import numpy as np                              # norms and vectorized margins.

t3_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t3_X = np.array([[2.0, 1.0], [1.0, 2.0], [2.0, 2.0], [-1.0, -1.0], [-2.0, -1.0], [-1.0, -2.0]])
t3_y = np.array([1.0, 1.0, 1.0, -1.0, -1.0, -1.0])
t3_w = np.array([1.0, 1.0])
t3_b = -0.5
t3_scores = t3_X @ t3_w                         # -> [3.0, 3.0, 4.0, -2.0, -3.0, -3.0]
t3_scores = t3_scores + t3_b                    # -> [2.5, 2.5, 3.5, -2.5, -3.5, -3.5]
t3_functional = t3_y * t3_scores                # -> [2.5, 2.5, 3.5, 2.5, 3.5, 3.5]
t3_norm = float(np.linalg.norm(t3_w))           # -> 1.4142135623730951
t3_geometric = t3_functional / t3_norm          # -> [1.768, 1.768, 2.475, 1.768, 2.475, 2.475]
t3_scale = 3.0
t3_scaled_scores = t3_X @ (t3_scale * t3_w)     # -> [9.0, 9.0, 12.0, -6.0, -9.0, -9.0]
t3_scaled_scores = t3_scaled_scores + t3_scale * t3_b  # -> [7.5, 7.5, 10.5, -7.5, -10.5, -10.5]
t3_scaled_functional = t3_y * t3_scaled_scores  # -> [7.5, 7.5, 10.5, 7.5, 10.5, 10.5]
t3_scaled_norm = float(np.linalg.norm(t3_scale * t3_w))  # -> 4.242640687119285
t3_scaled_geometric = t3_scaled_functional / t3_scaled_norm  # -> [1.768, 1.768, 2.475, 1.768, 2.475, 2.475]

print("functional margins:", t3_functional.tolist())  # -> [2.5, 2.5, 3.5, 2.5, 3.5, 3.5]
print("geometric margins:", np.round(t3_geometric, 3).tolist())  # -> [1.768, 1.768, 2.475, 1.768, 2.475, 2.475]
print("scaled functional:", t3_scaled_functional.tolist())  # -> [7.5, 7.5, 10.5, 7.5, 10.5, 10.5]
print("scaled geometric:", np.round(t3_scaled_geometric, 3).tolist())  # -> [1.768, 1.768, 2.475, 1.768, 2.475, 2.475]

assert np.allclose(t3_geometric, t3_scaled_geometric)

plt.figure(figsize=(4.8, 2.8))
plt.plot(np.arange(6), t3_geometric, marker="o", label="original")
plt.plot(np.arange(6), t3_scaled_geometric, marker="x", linestyle="--", label="scaled")
plt.xlabel("example")
plt.ylabel("geometric margin")
plt.legend()
plt.title("Toy 3 · distance survives score scaling")
plt.show()

▶ What you'll see: the scaled geometric-margin curve sits exactly on top of the original curve.

### ✍️ Toy 4 · Smaller norms mean wider streets

In canonical scaling, the one-sided margin width is `1 / ||w||`, so smaller norms mean wider margin
streets.

In [ ]:
import numpy as np                              # small arrays and argmax.

t4_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t4_norms = np.array([1.0, 1.5, 2.0, 0.5])
t4_widths = 1.0 / t4_norms                      # -> [1.0, 0.6666666666666666, 0.5, 2.0]
t4_streets = 2.0 / t4_norms                     # -> [2.0, 1.3333333333333333, 1.0, 4.0]
t4_objective = 0.5 * t4_norms ** 2              # -> [0.5, 1.125, 2.0, 0.125]
t4_best_width = int(np.argmax(t4_widths))       # -> 3

print("candidate ||w||:", t4_norms.tolist())   # -> [1.0, 1.5, 2.0, 0.5]
print("one-sided widths:", np.round(t4_widths, 3).tolist())  # -> [1.0, 0.667, 0.5, 2.0]
print("full street widths:", np.round(t4_streets, 3).tolist())  # -> [2.0, 1.333, 1.0, 4.0]
print("0.5||w||^2:", t4_objective.tolist())    # -> [0.5, 1.125, 2.0, 0.125]
print("widest candidate index:", t4_best_width)  # -> 3

assert t4_best_width == int(np.argmin(t4_objective))

plt.figure(figsize=(4.6, 2.8))
plt.bar(["w0", "w1", "w2", "w3"], t4_widths, color="purple")
plt.ylabel("1 / ||w||")
plt.title("Toy 4 · smallest norm gives widest margin")
plt.show()

▶ What you'll see: candidate `w3` has the smallest objective and the tallest margin-width bar.

### ✍️ Toy 5 · Hinge slack charges short margins

The soft-margin slack for each point is exactly `max(0, 1 - margin)`.

In [ ]:
import numpy as np                              # elementwise max.

t5_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t5_margins = np.array([1.4, 1.0, 0.6, 0.0, -0.5, 2.2])
t5_slack = np.maximum(0.0, 1.0 - t5_margins)    # -> [0.0, 0.0, 0.4, 1.0, 1.5, 0.0]
t5_total_slack = float(t5_slack.sum())          # -> 2.9

print("margins:", t5_margins.tolist())         # -> [1.4, 1.0, 0.6, 0.0, -0.5, 2.2]
print("slack values:", t5_slack.tolist())      # -> [0.0, 0.0, 0.4, 1.0, 1.5, 0.0]
print("total slack:", round(t5_total_slack, 3))  # -> 2.9

assert round(t5_total_slack, 3) == 2.9

t5_grid = np.linspace(-1.0, 2.5, 100)
t5_curve = np.maximum(0.0, 1.0 - t5_grid)
plt.figure(figsize=(4.6, 3.0))
plt.plot(t5_grid, t5_curve, color="navy")
plt.scatter(t5_margins, t5_slack, color="crimson", zorder=3)
plt.axvline(1.0, color="black", linestyle="--")
plt.xlabel("functional margin")
plt.ylabel("hinge / slack")
plt.title("Toy 5 · slack starts below margin 1")
plt.show()

▶ What you'll see: points with margin at least 1 pay zero, while the negative-margin point pays 1.5.

### ✍️ Toy 6 · C changes the winning objective

The soft-margin objective adds a norm term and `C` times total hinge loss. Changing `C` changes which
candidate is cheapest.

In [ ]:
import numpy as np                              # broadcasting and argmin.

t6_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t6_norm_terms = np.array([0.5, 1.2, 2.0])
t6_hinge_sums = np.array([3.0, 0.8, 0.1])
t6_Cs = np.array([0.1, 1.0, 5.0])
t6_objective = t6_norm_terms[None, :]           # -> [[0.5, 1.2, 2.0]]
t6_objective = t6_objective + t6_Cs[:, None] * t6_hinge_sums[None, :]  # -> [[0.8, 1.28, 2.01], [3.5, 2.0, 2.1], [15.5, 5.2, 2.5]]
t6_best = np.argmin(t6_objective, axis=1)       # -> [0, 1, 2]

print("norm terms:", t6_norm_terms.tolist())   # -> [0.5, 1.2, 2.0]
print("hinge sums:", t6_hinge_sums.tolist())   # -> [3.0, 0.8, 0.1]
print("C values:", t6_Cs.tolist())             # -> [0.1, 1.0, 5.0]
print("objectives:", np.round(t6_objective, 3).tolist())  # -> [[0.8, 1.28, 2.01], [3.5, 2.0, 2.1], [15.5, 5.2, 2.5]]
print("best candidate by C:", t6_best.tolist())  # -> [0, 1, 2]

assert t6_best.tolist() == [0, 1, 2]

plt.figure(figsize=(4.8, 3.0))
for t6_j in range(t6_objective.shape[1]):
    plt.plot(t6_Cs, t6_objective[:, t6_j], marker="o", label=f"cand {t6_j}")
plt.xscale("log")
plt.xlabel("C")
plt.ylabel("objective")
plt.legend()
plt.title("Toy 6 · bigger C rewards lower hinge")
plt.show()

▶ What you'll see: the winner moves from simple/high-hinge to complex/low-hinge as `C` grows.

### ✍️ Toy 7 · Support vectors are active constraints

Support vectors are the examples whose margins are at or below the target value 1.

In [ ]:
import numpy as np                              # masks and indices.

t7_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t7_X = np.array([[1.0, 1.0], [2.0, 2.0], [0.5, 0.4], [-1.0, -1.0], [-0.3, -0.2], [-2.0, -2.0]])
t7_y = np.array([1.0, 1.0, 1.0, -1.0, -1.0, -1.0])
t7_margins = np.array([1.0, 1.8, 0.7, 2.4, 1.0, 1.3])
t7_support_mask = t7_margins <= 1.0 + 1e-12     # -> [True, False, True, False, True, False]
t7_support_idx = np.where(t7_support_mask)[0]   # -> [0, 2, 4]

print("margins:", t7_margins.tolist())         # -> [1.0, 1.8, 0.7, 2.4, 1.0, 1.3]
print("support mask:", t7_support_mask.tolist())  # -> [True, False, True, False, True, False]
print("support indices:", t7_support_idx.tolist())  # -> [0, 2, 4]

assert t7_support_idx.tolist() == [0, 2, 4]

plt.figure(figsize=(4.6, 3.2))
plt.scatter(t7_X[t7_y == 1, 0], t7_X[t7_y == 1, 1], color="seagreen", label="+1")
plt.scatter(t7_X[t7_y == -1, 0], t7_X[t7_y == -1, 1], color="crimson", label="-1")
plt.scatter(t7_X[t7_support_mask, 0], t7_X[t7_support_mask, 1], s=180, facecolors="none", edgecolors="black", linewidths=2, label="support")
plt.axis("equal")
plt.legend()
plt.title("Toy 7 · circled points are active")
plt.show()

▶ What you'll see: exactly three circled points have margin 1 or less.

### ✍️ Toy 8 · Grid search selects the lowest soft-margin score

A tiny grid search tries a few one-dimensional separators and keeps the candidate with the smallest
soft-margin objective.

In [ ]:
import numpy as np                              # small grid search.

t8_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t8_X = np.array([-3.0, -2.0, -1.0, 1.0, 2.0, 3.0])
t8_y = np.array([-1.0, -1.0, -1.0, 1.0, 1.0, 1.0])
t8_weights = np.array([0.4, 0.8, 1.2])
t8_biases = np.array([-0.5, 0.0, 0.5])
t8_C = 0.5
t8_objective = np.zeros((t8_weights.size, t8_biases.size))
for t8_i, t8_weight in enumerate(t8_weights):
    for t8_j, t8_bias in enumerate(t8_biases):
        t8_margin = t8_y * (t8_weight * t8_X + t8_bias)
        t8_hinge = np.maximum(0.0, 1.0 - t8_margin)
        t8_objective[t8_i, t8_j] = 0.5 * t8_weight ** 2 + t8_C * t8_hinge.sum()
t8_best = np.unravel_index(np.argmin(t8_objective), t8_objective.shape)  # -> (1, 1)
t8_best_weight = float(t8_weights[t8_best[0]])  # -> 0.8
t8_best_bias = float(t8_biases[t8_best[1]])     # -> 0.0
t8_best_value = float(t8_objective[t8_best])    # -> 0.52

print("X:", t8_X.tolist())                     # -> [-3.0, -2.0, -1.0, 1.0, 2.0, 3.0]
print("y:", t8_y.astype(int).tolist())          # -> [-1, -1, -1, 1, 1, 1]
print("objective grid:", np.round(t8_objective, 3).tolist())  # -> [[1.18, 0.88, 1.18], [0.67, 0.52, 0.67], [0.87, 0.72, 0.87]]
print("best weight, bias, value:", t8_best_weight, t8_best_bias, round(t8_best_value, 3))  # -> 0.8 0.0 0.52

assert t8_best == (1, 1)

plt.figure(figsize=(4.2, 3.2))
plt.imshow(t8_objective, cmap="viridis", aspect="auto")
plt.colorbar(label="objective")
plt.xticks(range(t8_biases.size), t8_biases)
plt.yticks(range(t8_weights.size), t8_weights)
plt.scatter([t8_best[1]], [t8_best[0]], color="red", s=80)
plt.xlabel("bias")
plt.ylabel("weight")
plt.title("Toy 8 · red cell is the minimum")
plt.show()

▶ What you'll see: the center grid cell wins with objective `0.52`.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, dot products, norms, losses, and grid searches.
import matplotlib.pyplot as plt  # load Matplotlib for the margin, loss, and decision-boundary plots.
np.random.seed(0)  # make every example deterministic.

def svm_score(X, w, b):  # compute f(x)=w·x+b for a matrix of examples.
    return X @ w + b  # vectorized linear scores.

def functional_margin(X, y, w, b):  # compute y_i f(x_i) for every example.
    return y * svm_score(X, w, b)  # positive values mean correct-side predictions.

def hinge_loss_from_margin(margins):  # compute max(0,1-margin) elementwise.
    return np.maximum(0.0, 1.0 - margins)  # slack variables for the soft-margin objective.

def svm_objective(X, y, w, b, C):  # compute 0.5||w||^2 + C sum hinge.
    margins = functional_margin(X, y, w, b)  # get all functional margins.
    return 0.5 * float(np.dot(w, w)) + C * float(np.sum(hinge_loss_from_margin(margins)))  # full soft-margin score.

def plot_separator(X, y, w, b, title):  # draw a 2-D separator and its two margin lines.
    xs = np.linspace(np.min(X[:, 0]) - 1, np.max(X[:, 0]) + 1, 120)  # x-range for lines.
    plt.figure(figsize=(4.8, 3.7))  # compact figure.
    plt.scatter(X[y == 1, 0], X[y == 1, 1], color="seagreen", label="+1")  # positive class.
    plt.scatter(X[y == -1, 0], X[y == -1, 1], color="crimson", label="-1")  # negative class.
    if abs(w[1]) > 1e-12:  # avoid dividing by zero for vertical boundaries.
        plt.plot(xs, -(w[0] * xs + b) / w[1], color="black", label="decision")  # f=0.
        plt.plot(xs, (1 - b - w[0] * xs) / w[1], "--", color="gray", label="margins")  # f=+1.
        plt.plot(xs, (-1 - b - w[0] * xs) / w[1], "--", color="gray")  # f=-1.
    plt.axis("equal"); plt.legend(); plt.title(title); plt.show()  # display.

## 🟢 Basics (warm-up)

### Basic 1 — Score points with a linear separator

**Goal.** Compute $f(x)=w^\top x+b$, because every SVM decision starts as a signed linear score. We build it in 2 steps.

In [ ]:
X_b1 = np.array([[1., 1.], [2., 1.], [0., 0.], [-1., 0.]])  # two positive-looking and two negative-looking points.
y_b1 = np.array([1., 1., -1., -1.])  # labels as +1/-1 for margin calculations.
w_b1 = np.array([1., 1.])  # normal vector for a diagonal separator.
b_b1 = -1.0  # bias term that shifts the separator.

print("X_b1 shape:", X_b1.shape)  # inspect the mini dataset.

In [ ]:
scores_b1 = svm_score(X_b1, w_b1, b_b1)  # compute signed scores.
preds_b1 = np.sign(scores_b1)  # classify by sign.

print("scores:", scores_b1)  # inspect signed distance up to scale.
print("predictions:", preds_b1.astype(int))  # inspect predicted classes.

assert np.all(preds_b1 == y_b1)  # verify the line separates these four points.

▶ What you'll see: positive points have scores above zero and negative points below zero.

In [ ]:
xs_b1 = np.linspace(np.min(X_b1[:, 0]) - 1, np.max(X_b1[:, 0]) + 1, 120)  # x-range for the separator and margins.
yline_b1 = -(w_b1[0] * xs_b1 + b_b1) / w_b1[1]  # decision boundary f(x)=0.
upper_b1 = (1 - b_b1 - w_b1[0] * xs_b1) / w_b1[1]  # positive margin f(x)=+1.
lower_b1 = (-1 - b_b1 - w_b1[0] * xs_b1) / w_b1[1]  # negative margin f(x)=-1.
support_b1 = functional_margin(X_b1, y_b1, w_b1, b_b1) <= 1 + 1e-12  # points that touch the margin.
plt.figure(figsize=(4.8, 3.7))
plt.scatter(X_b1[y_b1 == 1, 0], X_b1[y_b1 == 1, 1], color="seagreen", label="+1")
plt.scatter(X_b1[y_b1 == -1, 0], X_b1[y_b1 == -1, 1], color="crimson", label="-1")
plt.scatter(X_b1[support_b1, 0], X_b1[support_b1, 1], s=170, facecolors="none", edgecolors="black", linewidths=2, label="support")
plt.plot(xs_b1, yline_b1, color="black", label="f(x)=0")
plt.plot(xs_b1, upper_b1, "--", color="gray", label="f(x)=±1")
plt.plot(xs_b1, lower_b1, "--", color="gray")
plt.axis("equal"); plt.legend(); plt.title("Basic 1: scores as separator geometry"); plt.show()

▶ What you'll see: the score signs match which side of the separating line each point sits on, with tight points circled on the margins.

👀 Takeaway: an SVM is linear at the scoring layer; the sign of $w^\top x+b$ is the predicted class.

### Basic 2 — Draw the decision boundary

**Goal.** Convert $w^\top x+b=0$ into a plotted line, because the margin is easiest to understand geometrically. We build it in 2 steps.

In [ ]:
X_b2 = np.array([[1., 1.], [2., 1.], [0., 0.], [-1., 0.]])  # reuse a small separable dataset.
y_b2 = np.array([1., 1., -1., -1.])  # class labels.
w_b2 = np.array([1., 1.])  # line normal.
b_b2 = -1.0  # line offset.
xs_b2 = np.linspace(-3, 4, 100)  # x-coordinates for drawing the line.

print("boundary equation: y = -(w0*x+b)/w1")  # state the rearranged formula.

In [ ]:
yline_b2 = -(w_b2[0] * xs_b2 + b_b2) / w_b2[1]  # solve the boundary equation for y.
plt.figure(figsize=(4.5, 3.4))  # create a compact decision plot.
plt.scatter(X_b2[y_b2 == 1, 0], X_b2[y_b2 == 1, 1], color="seagreen", label="+1")  # plot positives.
plt.scatter(X_b2[y_b2 == -1, 0], X_b2[y_b2 == -1, 1], color="crimson", label="-1")  # plot negatives.
plt.plot(xs_b2, yline_b2, color="black", label="f(x)=0")  # draw the separator.
plt.axis("equal"); plt.legend(); plt.title("Basic 2: decision boundary"); plt.show()  # display.

▶ What you'll see: a diagonal line with the two classes on opposite sides.

👀 Takeaway: the boundary is the zero-score contour of the linear function.

### Basic 3 — Compute functional margins

**Goal.** Multiply scores by labels, because $y_i f(x_i)$ measures whether each example is correct and how safely. We build it in 2 steps.

In [ ]:
X_b3 = np.array([[1., 1.], [2., 1.], [0., 0.], [-1., 0.]])  # toy data.
y_b3 = np.array([1., 1., -1., -1.])  # labels.
w_b3 = np.array([1., 1.])  # separator normal.
b_b3 = -1.0  # separator bias.
scores_b3 = svm_score(X_b3, w_b3, b_b3)  # raw scores before label multiplication.

print("raw scores:", scores_b3)  # inspect signed scores.

In [ ]:
margins_b3 = functional_margin(X_b3, y_b3, w_b3, b_b3)  # compute y*f(x).

print("functional margins:", margins_b3)  # inspect correctness margins.
print("minimum margin:", margins_b3.min())  # the tightest point matters most.

assert margins_b3.min() == 1.0  # verify canonical tightness.

▶ What you'll see: all functional margins are positive, so all four points are correctly classified.

In [ ]:
xs_b3 = np.linspace(np.min(X_b3[:, 0]) - 1, np.max(X_b3[:, 0]) + 1, 120)  # x-range for the margin geometry.
yline_b3 = -(w_b3[0] * xs_b3 + b_b3) / w_b3[1]  # decision boundary.
upper_b3 = (1 - b_b3 - w_b3[0] * xs_b3) / w_b3[1]  # +1 functional-margin line.
lower_b3 = (-1 - b_b3 - w_b3[0] * xs_b3) / w_b3[1]  # -1 functional-margin line.
support_b3 = margins_b3 <= 1 + 1e-12  # examples with the smallest functional margin.
plt.figure(figsize=(4.8, 3.7))
plt.scatter(X_b3[y_b3 == 1, 0], X_b3[y_b3 == 1, 1], color="seagreen", label="+1")
plt.scatter(X_b3[y_b3 == -1, 0], X_b3[y_b3 == -1, 1], color="crimson", label="-1")
plt.scatter(X_b3[support_b3, 0], X_b3[support_b3, 1], s=170, facecolors="none", edgecolors="black", linewidths=2, label="min margin")
plt.plot(xs_b3, yline_b3, color="black", label="f(x)=0")
plt.plot(xs_b3, upper_b3, "--", color="gray", label="margin lines")
plt.plot(xs_b3, lower_b3, "--", color="gray")
plt.axis("equal"); plt.legend(); plt.title("Basic 3: functional margins on the plane"); plt.show()

▶ What you'll see: points with functional margin 1 are circled exactly on the dashed canonical margin lines.

👀 Takeaway: functional margin puts both classes on one correctness scale.

### Basic 4 — Convert margin to distance

**Goal.** Divide by $\|w\|$, because geometric distance should not change when we rescale the score. We build it in 3 steps.

In [ ]:
X_b4 = np.array([[1., 1.], [0., 0.]])  # one positive and one negative point.
y_b4 = np.array([1., -1.])  # labels.
w_b4 = np.array([1., 1.])  # separator normal.
b_b4 = -1.0  # separator bias.
functional_b4 = functional_margin(X_b4, y_b4, w_b4, b_b4)  # raw functional margins.

print("functional margins:", functional_b4)  # inspect scale-dependent values.

In [ ]:
norm_b4 = np.linalg.norm(w_b4)  # length of the normal vector.
distance_b4 = functional_b4 / norm_b4  # signed geometric distance to the boundary.

print("||w||:", round(norm_b4, 3))  # inspect denominator.
print("geometric distances:", np.round(distance_b4, 3))  # inspect actual distances.

assert round(float(distance_b4.min()), 3) == 0.707  # verify the worked distance.

In [ ]:
plt.figure(figsize=(4, 3))  # make a small distance plot.
plt.bar(["pos", "neg"], distance_b4, color=["seagreen", "crimson"])  # show distances by point.
plt.ylabel("y f(x) / ||w||")  # label geometric margin.
plt.title("Basic 4: geometric margin")  # title the plot.
plt.show()  # display.

▶ What you'll see: both example distances are positive, with the closer one at about 0.707.

👀 Takeaway: geometric margin is the functional margin corrected for the length of $w$.

### Basic 5 — Verify scaling invariance

**Goal.** Scale $w$ and $b$ together, because it reveals why SVMs need geometric margins. We build it in 2 steps.

In [ ]:
X_b5 = np.array([[1., 1.], [0., 0.]])  # simple classified points.
y_b5 = np.array([1., -1.])  # labels.
w_b5 = np.array([1., 1.])  # original normal.
b_b5 = -1.0  # original bias.
scale_b5 = 7.0  # arbitrary positive rescaling.

print("scale:", scale_b5)  # inspect multiplier.

In [ ]:
m1_b5 = functional_margin(X_b5, y_b5, w_b5, b_b5)  # original functional margin.
m2_b5 = functional_margin(X_b5, y_b5, scale_b5 * w_b5, scale_b5 * b_b5)  # scaled functional margin.
g1_b5 = m1_b5 / np.linalg.norm(w_b5)  # original geometric margin.
g2_b5 = m2_b5 / np.linalg.norm(scale_b5 * w_b5)  # scaled geometric margin.

print("functional original/scaled:", m1_b5, m2_b5)  # functional margins inflate.
print("geometric original/scaled:", np.round(g1_b5, 3), np.round(g2_b5, 3))  # distances match.

assert np.allclose(g1_b5, g2_b5)  # verify invariant geometry.

▶ What you'll see: functional margins multiply by 7, but geometric margins stay unchanged.

In [ ]:
xs_b5 = np.linspace(-1.5, 2.0, 120)  # shared x-range for original and scaled separators.
yline_b5 = -(w_b5[0] * xs_b5 + b_b5) / w_b5[1]  # original boundary.
yline_scaled_b5 = -((scale_b5 * w_b5)[0] * xs_b5 + scale_b5 * b_b5) / (scale_b5 * w_b5)[1]  # scaled boundary.
upper_b5 = (1 - b_b5 - w_b5[0] * xs_b5) / w_b5[1]  # original +1 margin.
lower_b5 = (-1 - b_b5 - w_b5[0] * xs_b5) / w_b5[1]  # original -1 margin.
support_b5 = g1_b5 <= np.min(g1_b5) + 1e-12  # closest points by geometric distance.
plt.figure(figsize=(4.8, 3.7))
plt.scatter(X_b5[y_b5 == 1, 0], X_b5[y_b5 == 1, 1], color="seagreen", label="+1")
plt.scatter(X_b5[y_b5 == -1, 0], X_b5[y_b5 == -1, 1], color="crimson", label="-1")
plt.scatter(X_b5[support_b5, 0], X_b5[support_b5, 1], s=170, facecolors="none", edgecolors="black", linewidths=2, label="closest")
plt.plot(xs_b5, yline_b5, color="black", label="original f=0")
plt.plot(xs_b5, yline_scaled_b5, ":", color="purple", linewidth=3, label="scaled f=0")
plt.plot(xs_b5, upper_b5, "--", color="gray", label="original margins")
plt.plot(xs_b5, lower_b5, "--", color="gray")
plt.axis("equal"); plt.legend(); plt.title("Basic 5: scaling leaves geometry fixed"); plt.show()

▶ What you'll see: the scaled boundary lies directly on top of the original boundary, even though the raw functional margins changed.

👀 Takeaway: the real separator geometry is unchanged by positive rescaling of all scores.

### Basic 6 — Plot the margin lines

**Goal.** Draw $f(x)=0$, $f(x)=1$, and $f(x)=-1$, because the SVM margin is a street around the boundary. We build it in 2 steps.

In [ ]:
X_b6 = np.array([[1., 1.], [2., 1.], [1., 2.], [0., 0.], [-1., 0.], [0., -1.]])  # separable toy data.
y_b6 = np.array([1., 1., 1., -1., -1., -1.])  # labels.
w_b6 = np.array([1., 1.])  # separator normal.
b_b6 = -1.0  # bias.

print("margin half-width:", round(1 / np.linalg.norm(w_b6), 3))  # distance from boundary to either margin line.

assert round(1 / np.linalg.norm(w_b6), 3) == 0.707  # verify half-width.

In [ ]:
plot_separator(X_b6, y_b6, w_b6, b_b6, "Basic 6: boundary and margins")  # draw decision and margin lines.

▶ What you'll see: two dashed margin lines parallel to the decision boundary.

👀 Takeaway: the SVM tries to make this margin street as wide as constraints allow.

### Basic 7 — Compute hinge loss

**Goal.** Apply $\max(0,1-yf(x))$, because soft-margin SVMs charge only points that fail the desired margin. We build it in 2 steps.

In [ ]:
margins_b7 = np.array([2.5, 1.0, 0.4, -0.5])  # examples: safe, exactly on margin, inside margin, misclassified.
hinge_b7 = hinge_loss_from_margin(margins_b7)  # compute slack/hinge values.

print("margins:", margins_b7)  # inspect inputs.
print("hinge losses:", hinge_b7)  # inspect violations.

assert np.allclose(hinge_b7, [0.0, 0.0, 0.6, 1.5])  # verify hinge arithmetic.

In [ ]:
plt.figure(figsize=(4, 3))  # create hinge bar chart.
plt.bar(["safe", "on", "inside", "wrong"], hinge_b7, color="darkorange")  # show per-case penalties.
plt.ylabel("max(0, 1-margin)")  # label loss.
plt.title("Basic 7: hinge loss cases")  # title.
plt.show()  # display.

▶ What you'll see: safe and exactly-on-margin points have zero hinge loss; violations pay positive loss.

👀 Takeaway: hinge loss is the slack required to make a margin constraint feasible.

### Basic 8 — Identify support vectors

**Goal.** Mark points with margin at most 1, because those examples touch or violate the margin and can determine the boundary. We build it in 2 steps.

In [ ]:
X_b8 = np.array([[1., 1.], [2., 1.], [1., 2.], [0., 0.], [-1., 0.], [0., -1.]])  # toy data.
y_b8 = np.array([1., 1., 1., -1., -1., -1.])  # labels.
w_b8 = np.array([1., 1.])  # separator normal.
b_b8 = -1.0  # separator bias.
margins_b8 = functional_margin(X_b8, y_b8, w_b8, b_b8)  # compute margins.

print("margins:", margins_b8)  # inspect tightness.

In [ ]:
support_b8 = margins_b8 <= 1 + 1e-12  # support vectors are active or violating constraints.

print("support indices:", np.where(support_b8)[0])  # inspect selected points.

assert np.where(support_b8)[0].tolist() == [0, 3]  # verify the two canonical support vectors.
plot_separator(X_b8, y_b8, w_b8, b_b8, "Basic 8: support-vector candidates")  # draw separator for context.

▶ What you'll see: points closest to the dashed margin lines are the support vectors.

👀 Takeaway: far-away correctly classified points usually do not control the SVM solution.

### Basic 9 — Compute the soft-margin objective

**Goal.** Add the norm penalty and hinge penalties, because SVM model selection uses the full objective rather than raw accuracy. We build it in 3 steps.

In [ ]:
X_b9 = np.array([[1., 1.], [0.2, 0.1], [0., 0.], [-1., 0.]])  # include one difficult positive point.
y_b9 = np.array([1., 1., -1., -1.])  # labels.
w_b9 = np.array([1., 1.])  # candidate separator.
b_b9 = -1.0  # candidate bias.
C_b9 = 1.0  # violation cost.

print("C:", C_b9)  # inspect the cost knob.

In [ ]:
margins_b9 = functional_margin(X_b9, y_b9, w_b9, b_b9)  # compute margins.
hinge_b9 = hinge_loss_from_margin(margins_b9)  # compute violations.

print("margins:", np.round(margins_b9, 3))  # inspect each constraint.
print("hinge:", np.round(hinge_b9, 3))  # inspect slack values.

assert round(float(hinge_b9[1]), 3) == 1.7  # verify hard point loss.

In [ ]:
obj_b9 = svm_objective(X_b9, y_b9, w_b9, b_b9, C_b9)  # compute 0.5||w||² + CΣhinge.

print("objective:", round(obj_b9, 3))  # inspect full score.

assert round(obj_b9, 3) == 2.7  # 0.5*2 + 1.7.

▶ What you'll see: the full objective combines a complexity cost of 1.0 with a hinge cost of 1.7.

In [ ]:
xs_b9 = np.linspace(np.min(X_b9[:, 0]) - 1, np.max(X_b9[:, 0]) + 1, 120)  # x-range for the soft-margin geometry.
yline_b9 = -(w_b9[0] * xs_b9 + b_b9) / w_b9[1]  # decision boundary.
upper_b9 = (1 - b_b9 - w_b9[0] * xs_b9) / w_b9[1]  # positive margin line.
lower_b9 = (-1 - b_b9 - w_b9[0] * xs_b9) / w_b9[1]  # negative margin line.
violating_b9 = hinge_b9 > 0  # points inside the margin or misclassified.
plt.figure(figsize=(4.8, 3.7))
plt.scatter(X_b9[y_b9 == 1, 0], X_b9[y_b9 == 1, 1], color="seagreen", label="+1")
plt.scatter(X_b9[y_b9 == -1, 0], X_b9[y_b9 == -1, 1], color="crimson", label="-1")
plt.scatter(X_b9[violating_b9, 0], X_b9[violating_b9, 1], s=190, facecolors="none", edgecolors="darkorange", linewidths=2.5, label="hinge > 0")
plt.plot(xs_b9, yline_b9, color="black", label="f(x)=0")
plt.plot(xs_b9, upper_b9, "--", color="gray", label="margin lines")
plt.plot(xs_b9, lower_b9, "--", color="gray")
plt.axis("equal"); plt.legend(); plt.title("Basic 9: objective sees margin violations"); plt.show()

▶ What you'll see: the difficult positive point is circled inside the wrong side of the margin, explaining the hinge cost in the objective.

👀 Takeaway: SVM training trades off margin width and violations in one scalar objective.

### Basic 10 — Compare two C values

**Goal.** Change $C$, because this single knob controls how much the learner cares about margin violations. We build it in 2 steps.

In [ ]:
X_b10 = np.array([[1., 1.], [0.2, 0.1], [0., 0.], [-1., 0.]])  # same soft-margin data.
y_b10 = np.array([1., 1., -1., -1.])  # labels.
w_b10 = np.array([1., 1.])  # candidate separator.
b_b10 = -1.0  # candidate bias.
Cs_b10 = np.array([0.1, 1.0, 10.0])  # forgiving, moderate, strict penalties.

print("C values:", Cs_b10)  # inspect sweep.

In [ ]:
objs_b10 = np.array([svm_objective(X_b10, y_b10, w_b10, b_b10, C_b10) for C_b10 in Cs_b10])  # objective for each C.

print("objectives:", np.round(objs_b10, 3))  # inspect how violation cost scales.

assert np.allclose(np.round(objs_b10, 2), [1.17, 2.70, 18.00])  # verify objective values.
plt.figure(figsize=(4, 3))  # create C sweep chart.
plt.plot(Cs_b10, objs_b10, marker="o", color="purple")  # objective versus C.
plt.xscale("log"); plt.xlabel("C"); plt.ylabel("objective")  # log-scale C.
plt.title("Basic 10: C scales hinge cost")  # title.
plt.show()  # display.

▶ What you'll see: the same separator becomes much more expensive when C is large.

👀 Takeaway: large C pressures the model to reduce violations; small C tolerates them to preserve simplicity.

## 🟡 Easy

### Easy 1 — Choose the widest hard-margin separator

**Goal.** Evaluate several separating candidates, because hard-margin SVM picks the feasible separator with the largest geometric margin. We build it in 3 steps.

In [ ]:
X_e1 = np.array([[1., 1.], [2., 1.], [1., 2.], [0., 0.], [-1., 0.], [0., -1.]])  # separable data.
y_e1 = np.array([1., 1., 1., -1., -1., -1.])  # labels.
candidates_e1 = [(np.array([1., 1.]), -1.0), (np.array([2., 1.]), -1.0), (np.array([1., 2.]), -1.0)]  # candidate lines.

print("candidate count:", len(candidates_e1))  # inspect alternatives.

In [ ]:
min_geo_e1 = []  # store minimum geometric margin for each feasible candidate.
for w_e1, b_e1 in candidates_e1:
    margins_e1 = functional_margin(X_e1, y_e1, w_e1, b_e1)  # compute functional margins.
    geo_e1 = margins_e1 / np.linalg.norm(w_e1)  # convert to distances.
    min_geo_e1.append(float(np.min(geo_e1)))  # keep the worst-case distance.

print("minimum geometric margins:", np.round(min_geo_e1, 3))  # inspect margin widths.

In [ ]:
best_e1 = int(np.argmax(min_geo_e1))  # choose largest minimum distance.

print("best candidate:", best_e1)  # inspect selected separator.

assert best_e1 == 0  # the diagonal candidate has the widest margin among these.
plt.figure(figsize=(4, 3))
plt.bar(["cand0", "cand1", "cand2"], min_geo_e1, color="seagreen")
plt.ylabel("minimum geometric margin"); plt.title("Easy 1: hard-margin choice"); plt.show()

▶ What you'll see: candidate 0 has the largest worst-case geometric margin.

👀 Takeaway: hard-margin SVM is a max-min geometry problem over feasible separating lines.

### Easy 2 — Use hinge loss on a nonseparable point

**Goal.** Compute slack for one difficult point, because soft-margin SVMs must handle imperfect separation. We build it in 3 steps.

In [ ]:
X_e2 = np.array([[1., 1.], [0.2, 0.1], [0., 0.], [-1., 0.]])  # one positive point is difficult.
y_e2 = np.array([1., 1., -1., -1.])  # labels.
w_e2 = np.array([1., 1.])  # candidate separator.
b_e2 = -1.0  # bias.

print("scores:", np.round(svm_score(X_e2, w_e2, b_e2), 3))  # inspect raw scores.

In [ ]:
margins_e2 = functional_margin(X_e2, y_e2, w_e2, b_e2)  # compute y*f(x).
slack_e2 = hinge_loss_from_margin(margins_e2)  # compute slack values.

print("margins:", np.round(margins_e2, 3))  # inspect margin violations.
print("slack:", np.round(slack_e2, 3))  # inspect hinge losses.

assert round(float(np.sum(slack_e2)), 3) == 1.7  # total violation in this toy.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(np.arange(len(slack_e2)), slack_e2, color=np.where(slack_e2 > 0, "crimson", "gray"))
plt.xlabel("point index"); plt.ylabel("slack ξ"); plt.title("Easy 2: soft-margin violations"); plt.show()

▶ What you'll see: only the difficult point pays slack.

👀 Takeaway: slack variables measure exactly how far examples fall short of the unit margin.

### Easy 3 — Grid search a soft-margin objective

**Goal.** Search a tiny parameter grid, because it makes the SVM objective concrete without a specialized solver. We build it in 4 steps.

In [ ]:
X_e3 = np.array([[1., 1.], [2., 1.], [0.2, 0.1], [0., 0.], [-1., 0.]])  # soft-margin toy data.
y_e3 = np.array([1., 1., 1., -1., -1.])  # labels.
w_grid_e3 = [np.array([0.6, 0.6]), np.array([1.0, 1.0]), np.array([1.4, 1.4])]  # candidate normals.
b_grid_e3 = [-1.4, -1.0, -0.6]  # candidate biases.
C_e3 = 1.0  # moderate violation cost.

print("grid size:", len(w_grid_e3) * len(b_grid_e3))  # inspect candidate count.

In [ ]:
best_obj_e3 = np.inf  # initialize best objective.
best_w_e3 = None  # initialize best normal.
best_b_e3 = None  # initialize best bias.
for w_e3 in w_grid_e3:
    for b_e3 in b_grid_e3:
        obj_e3 = svm_objective(X_e3, y_e3, w_e3, b_e3, C_e3)  # score candidate.
        if obj_e3 < best_obj_e3:
            best_obj_e3, best_w_e3, best_b_e3 = obj_e3, w_e3, b_e3  # keep best.

print("best:", best_w_e3, best_b_e3, round(best_obj_e3, 3))  # inspect winner.

In [ ]:
margins_e3 = functional_margin(X_e3, y_e3, best_w_e3, best_b_e3)  # margins for best candidate.

print("best margins:", np.round(margins_e3, 3))  # inspect active constraints.

assert round(best_obj_e3, 3) == 2.58  # verify this grid's selected objective.

In [ ]:
plot_separator(X_e3, y_e3, best_w_e3, best_b_e3, "Easy 3: best grid separator")  # visualize chosen line.

▶ What you'll see: the grid-selected separator balances margin size and the hard positive point's violation.

👀 Takeaway: even a crude search chooses by full objective, not by accuracy alone.

### Easy 4 — Compare raw loss plus cost numbers

**Goal.** Reproduce the lesson's empirical-score arithmetic, because model choice uses raw fit plus a cost term. We build it in 3 steps.

In [ ]:
losses_e4 = np.array([0.202, 0.122, 0.454])  # verified per-example losses from the lesson prose.
cost_e4 = 0.100  # method cost / regularization guardrail.
alt_e4 = 0.411  # more flexible alternative's decision score.

print("losses:", losses_e4)  # inspect raw terms.

In [ ]:
risk_e4 = float(np.mean(losses_e4))  # empirical risk average.
score_e4 = risk_e4 + cost_e4  # decision score includes cost.
gap_e4 = alt_e4 - score_e4  # evidence gap versus alternative.
rel_gap_e4 = gap_e4 / alt_e4  # relative gap scale.

print("risk:", round(risk_e4, 3), "score:", round(score_e4, 3))  # inspect full score.
print("gap:", round(gap_e4, 3), "relative:", round(rel_gap_e4, 3))  # inspect comparison.

assert round(risk_e4, 3) == 0.259 and round(score_e4, 3) == 0.359  # verify lesson values.

In [ ]:
stable_e4 = 0.80 * score_e4  # stabilizing knob reduces decision score by 20 percent.
choices_e4 = np.array([score_e4, alt_e4, stable_e4])  # baseline, flexible alternative, stabilized score.

print("stable score:", round(stable_e4, 3), "best:", round(float(np.min(choices_e4)), 3))  # inspect final decision.

assert round(stable_e4, 3) == 0.287  # verify lesson value.
plt.figure(figsize=(4, 3)); plt.bar(["base", "flex", "stable"], choices_e4, color=["gray", "orange", "seagreen"])
plt.ylabel("decision score (lower better)"); plt.title("Easy 4: score + cost decision"); plt.show()

▶ What you'll see: the stabilized score is the lowest of the three choices.

👀 Takeaway: SVM selection should compare complete objective-like scores, not raw fit alone.

### Easy 5 — Visualize support-vector sensitivity

**Goal.** Move a far point and a support vector, because only active margin points should change the boundary pressure. We build it in 3 steps.

In [ ]:
X_e5 = np.array([[1., 1.], [2., 1.], [1., 2.], [0., 0.], [-1., 0.], [0., -1.]])  # separable data.
y_e5 = np.array([1., 1., 1., -1., -1., -1.])  # labels.
w_e5 = np.array([1., 1.])  # separator.
b_e5 = -1.0  # bias.
margins_e5 = functional_margin(X_e5, y_e5, w_e5, b_e5)  # original margins.

print("original margins:", margins_e5)  # inspect support and non-support points.

In [ ]:
X_far_e5 = X_e5.copy(); X_far_e5[1] += np.array([2., 2.])  # move a far positive farther away.
X_support_e5 = X_e5.copy(); X_support_e5[0] += np.array([-0.6, -0.6])  # move a support vector toward the boundary.
far_min_e5 = functional_margin(X_far_e5, y_e5, w_e5, b_e5).min()  # minimum margin after moving far point.
support_min_e5 = functional_margin(X_support_e5, y_e5, w_e5, b_e5).min()  # minimum margin after moving support vector.

print("min after far move:", far_min_e5, "min after support move:", round(float(support_min_e5), 3))  # compare effects.

assert far_min_e5 == 1.0 and round(float(support_min_e5), 3) == -0.2  # verify sensitivity.

In [ ]:
plt.figure(figsize=(4.8, 3.6))
plt.scatter(X_e5[:, 0], X_e5[:, 1], c=np.where(y_e5 > 0, "seagreen", "crimson"), label="original")
plt.scatter(X_support_e5[0:1, 0], X_support_e5[0:1, 1], marker="x", s=120, color="black", label="moved support")
plt.title("Easy 5: support point movement matters"); plt.legend(); plt.axis("equal"); plt.show()

▶ What you'll see: moving the support point changes the minimum margin, while moving a far point does not.

👀 Takeaway: support vectors are influential because they sit on active or violated constraints.

## 🔴 Advanced

### Advanced 1 — Sweep C and validation error

**Goal.** Compare train objective and validation mistakes across C values, because the violation penalty is a regularization knob. We build it in 4 steps.

In [ ]:
X_a1 = np.array([[1., 1.], [2., 1.], [0.2, 0.1], [0., 0.], [-1., 0.], [1.0, 0.2]])  # training points with one difficult positive.
y_a1 = np.array([1., 1., 1., -1., -1., -1.])  # labels include a tricky negative near the boundary.
X_val_a1 = np.array([[1.5, 1.0], [-0.5, 0.0], [0.5, 0.6]])  # validation points.
y_val_a1 = np.array([1., -1., -1.])  # validation labels.
Cs_a1 = np.array([0.1, 1.0, 10.0])  # C values to compare.

print("C sweep:", Cs_a1)  # inspect grid.

In [ ]:
candidates_a1 = [(np.array([0.6, 0.6]), -0.6), (np.array([1.0, 1.0]), -1.0), (np.array([1.6, 1.6]), -1.0)]  # separators from simple to strict.
best_objs_a1 = []  # store training objective.
val_errs_a1 = []  # store validation error.
chosen_a1 = []  # store chosen candidate index.
for C_a1 in Cs_a1:
    objs_a1 = np.array([svm_objective(X_a1, y_a1, w_a1, b_a1, C_a1) for w_a1, b_a1 in candidates_a1])  # train scores.
    idx_a1 = int(np.argmin(objs_a1))  # choose by objective.
    w_best_a1, b_best_a1 = candidates_a1[idx_a1]  # selected model.
    err_a1 = np.mean(np.not_equal(np.sign(svm_score(X_val_a1, w_best_a1, b_best_a1)), y_val_a1))  # validation 0/1 error.
    best_objs_a1.append(float(objs_a1[idx_a1])); val_errs_a1.append(float(err_a1)); chosen_a1.append(idx_a1)  # store.

print("chosen candidates:", chosen_a1)  # inspect model choices.
print("validation errors:", val_errs_a1)  # inspect generalization proxy.

In [ ]:
assert len(chosen_a1) == 3 and np.all(np.array(best_objs_a1) > 0)  # sanity-check sweep values.
plt.figure(figsize=(5, 3))
plt.plot(Cs_a1, best_objs_a1, marker="o", label="best train objective")
plt.plot(Cs_a1, val_errs_a1, marker="s", label="validation error")
plt.xscale("log"); plt.xlabel("C"); plt.title("Advanced 1: C sweep"); plt.legend(); plt.show()

▶ What you'll see: changing C can change which separator is selected and how validation error behaves.

👀 Takeaway: C should be selected by held-out behavior, not by assuming stricter training fit is always better.

### Advanced 2 — Build a polynomial feature map by hand

**Goal.** Separate a nonlinear pattern with explicit features, because a linear SVM can be linear in transformed space. We build it in 4 steps.

In [ ]:
X_a2 = np.array([[0., 0.], [0.2, -0.1], [2., 0.], [-2., 0.], [0., 2.], [0., -2.]])  # center points vs outer ring.
y_a2 = np.array([-1., -1., 1., 1., 1., 1.])  # outer points positive, center points negative.

print("raw shape:", X_a2.shape)  # inspect original 2-D data.

In [ ]:
Phi_a2 = np.column_stack([X_a2[:, 0], X_a2[:, 1], X_a2[:, 0] ** 2 + X_a2[:, 1] ** 2])  # explicit radius feature.
w_a2 = np.array([0., 0., 1.])  # classify by radius only.
b_a2 = -1.0  # threshold radius squared at 1.
margins_a2 = functional_margin(Phi_a2, y_a2, w_a2, b_a2)  # margins in feature space.

print("feature rows:\n", Phi_a2)  # inspect transformed coordinates.
print("margins:", np.round(margins_a2, 3))  # inspect separation.

assert np.all(margins_a2 > 0)  # transformed linear separator works.

In [ ]:
xx_a2 = np.linspace(-2.5, 2.5, 120); yy_a2 = np.linspace(-2.5, 2.5, 120)  # grid for circular boundary.
GX_a2, GY_a2 = np.meshgrid(xx_a2, yy_a2)  # 2-D grid.
Gscore_a2 = GX_a2 ** 2 + GY_a2 ** 2 - 1  # score from radius feature.
plt.figure(figsize=(4.4, 4))
plt.contour(GX_a2, GY_a2, Gscore_a2, levels=[0], colors="black")
plt.scatter(X_a2[y_a2 == 1, 0], X_a2[y_a2 == 1, 1], color="seagreen", label="+1")
plt.scatter(X_a2[y_a2 == -1, 0], X_a2[y_a2 == -1, 1], color="crimson", label="-1")
plt.axis("equal"); plt.legend(); plt.title("Advanced 2: linear after feature map"); plt.show()

▶ What you'll see: the linear rule in radius-squared space appears as a circle in the original 2-D plot.

👀 Takeaway: kernels later automate this idea, but the core SVM still maximizes margins in a feature space.

### Advanced 3 — Approximate subgradient descent on hinge loss

**Goal.** Train a tiny linear SVM with subgradient steps, because hinge loss is piecewise linear rather than squared. We build it in 5 steps.

In [ ]:
X_a3 = np.array([[1., 1.], [2., 1.], [0., 0.], [-1., 0.]])  # separable tiny data.
y_a3 = np.array([1., 1., -1., -1.])  # labels.
w_a3 = np.zeros(2)  # start with no direction.
b_a3 = 0.0  # start with no offset.
eta_a3 = 0.05  # learning rate.
C_a3 = 1.0  # hinge cost.

print("initial w,b:", w_a3, b_a3)  # inspect initialization.

In [ ]:
losses_a3 = []  # track objective values.
for epoch_a3 in range(80):
    for i_a3 in range(len(y_a3)):
        margin_a3 = y_a3[i_a3] * (X_a3[i_a3] @ w_a3 + b_a3)  # current margin.
        grad_w_a3 = w_a3.copy()  # derivative of 0.5||w||².
        grad_b_a3 = 0.0  # no regularization on bias.
        if margin_a3 < 1:  # hinge is active.
            grad_w_a3 -= C_a3 * y_a3[i_a3] * X_a3[i_a3]  # subgradient of hinge.
            grad_b_a3 -= C_a3 * y_a3[i_a3]  # bias subgradient.
        w_a3 -= eta_a3 * grad_w_a3  # descend in w.
        b_a3 -= eta_a3 * grad_b_a3  # descend in b.
    losses_a3.append(svm_objective(X_a3, y_a3, w_a3, b_a3, C_a3))  # record objective.

print("trained w,b:", np.round(w_a3, 3), round(b_a3, 3))  # inspect learned separator.
print("loss start/end:", round(losses_a3[0], 3), round(losses_a3[-1], 3))  # inspect convergence.

assert losses_a3[-1] < losses_a3[0]  # verify training improved objective.

In [ ]:
margins_a3 = functional_margin(X_a3, y_a3, w_a3, b_a3)  # final margins.

print("final margins:", np.round(margins_a3, 3))  # inspect constraint status.

assert np.all(margins_a3 > 0)  # the trained separator classifies the toy set correctly.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(losses_a3, color="purple")
plt.xlabel("epoch"); plt.ylabel("objective"); plt.title("Advanced 3: hinge subgradient training"); plt.show()

▶ What you'll see: the objective decreases and then flattens as the separating direction stabilizes.

👀 Takeaway: SVM solvers follow gradients or dual updates, but the pressure is always norm shrinkage plus active hinge violations.

### Advanced 4 — Measure margin width after training

**Goal.** Compute support vectors and margin width from the trained subgradient model, because the learned parameters should be interpreted geometrically. We build it in 3 steps.

In [ ]:
X_a4 = X_a3.copy()  # reuse the trained-data coordinates from Advanced 3.
y_a4 = y_a3.copy()  # reuse labels.
w_a4 = w_a3.copy()  # reuse trained weights.
b_a4 = b_a3  # reuse trained bias.

print("trained norm:", round(float(np.linalg.norm(w_a4)), 3))  # inspect normal length.

In [ ]:
margins_a4 = functional_margin(X_a4, y_a4, w_a4, b_a4)  # functional margins.
geo_a4 = margins_a4 / np.linalg.norm(w_a4)  # geometric distances.
support_a4 = margins_a4 <= 1.05  # approximately active support vectors after SGD.

print("geometric margins:", np.round(geo_a4, 3))  # inspect distances.
print("approx support indices:", np.where(support_a4)[0])  # inspect active points.

assert float(np.min(geo_a4)) > 0  # all examples lie on the correct side.

In [ ]:
plot_separator(X_a4, y_a4, w_a4, b_a4, "Advanced 4: trained margin geometry")  # visualize learned separator.

▶ What you'll see: the learned line separates the points and the closest points sit near the margin bands.

👀 Takeaway: after training, $\|w\|$ and the active margins tell the geometric story of the classifier.

### Advanced 5 — Compare SVM objective with accuracy

**Goal.** Show two models with the same accuracy but different margins, because SVMs optimize a margin objective rather than only counting mistakes. We build it in 4 steps.

In [ ]:
X_a5 = np.array([[1., 1.], [2., 1.], [1., 2.], [0., 0.], [-1., 0.], [0., -1.]])  # separable data.
y_a5 = np.array([1., 1., 1., -1., -1., -1.])  # labels.
models_a5 = [(np.array([1., 1.]), -1.0), (np.array([3., 3.]), -3.0)]  # same boundary direction, different scale.

print("model count:", len(models_a5))  # inspect alternatives.

In [ ]:
accs_a5 = []  # store accuracies.
objs_a5 = []  # store objectives.
widths_a5 = []  # store margin half-widths.
for w_a5, b_a5 in models_a5:
    pred_a5 = np.sign(svm_score(X_a5, w_a5, b_a5))  # predictions.
    accs_a5.append(float(np.mean(pred_a5 == y_a5)))  # accuracy.
    objs_a5.append(svm_objective(X_a5, y_a5, w_a5, b_a5, C=1.0))  # SVM objective.
    widths_a5.append(float(1 / np.linalg.norm(w_a5)))  # margin half-width under canonical bands.

print("accuracies:", accs_a5)  # both classify perfectly.
print("objectives:", np.round(objs_a5, 3), "widths:", np.round(widths_a5, 3))  # objective distinguishes them.

assert accs_a5[0] == accs_a5[1] == 1.0  # same classification accuracy.

In [ ]:
plt.figure(figsize=(5, 3))
x_a5 = np.arange(2)  # bar positions.
plt.bar(x_a5 - 0.18, objs_a5, width=0.36, label="objective", color="orange")
plt.bar(x_a5 + 0.18, widths_a5, width=0.36, label="margin half-width", color="seagreen")
plt.xticks(x_a5, ["model 0", "model 1"]); plt.title("Advanced 5: accuracy is not enough"); plt.legend(); plt.show()

▶ What you'll see: both models have 100 percent accuracy, but the larger-norm model has a worse objective and narrower margin.

👀 Takeaway: SVMs prefer separators that are not just correct, but correct with the widest justified margin.